# Week 4 Thursday Exercise 2 — Fraud Propensity Decision Tree

This notebook builds a simple propensity model for synthetic identity fraud.


In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text


In [2]:
df = pd.read_csv('week4_thursday_fraud_propensity_dataset.csv')
df


,application_id,customer_name,application_type,file_depth,account_age,identity_consistency,phone_reuse,email_reuse,address_reuse,device_reuse,ip_risk,synthetic_indicator,manual_risk_label
0,T001,Aster Bank,New,Thin-file,<1 mo,Mismatched,Reused,Reused,Reused,Shared,High,Yes,High
1,T002,Blue Harbor Retail,New,Stable,1-12 mo,Consistent,Same,Same,Same,Same,Low,No,Low
2,T003,Cedar Lending,New,Thin-file,<1 mo,Partial,Shared,Reused,Same,Shared,High,Yes,High
3,T004,Delta Credit,Existing,Moderate,12+ mo,Consistent,Same,Same,Same,Same,Low,No,Low
4,T005,Echo Finance,New,Thin-file,<1 mo,Mismatched,Shared,Shared,Reused,Shared,High,Yes,High
5,T006,Fjord Payments,New,Thin-file,<1 mo,Partial,Shared,Same,Reused,Shared,High,Yes,High
6,T007,Grove Markets,Existing,Stable,12+ mo,Consistent,Same,Shared,Same,Same,Low,No,Low
7,T008,Harbor Loans,New,Thin-file,<1 mo,Mismatched,Reused,Shared,Reused,Shared,High,Yes,High


## Define the target
High labels are treated as positive synthetic-risk cases.


In [3]:
df['target'] = df['manual_risk_label'].map({'Low':0,'High':1}).fillna(0).astype(int)
df[['application_id','customer_name','manual_risk_label','target']]


,application_id,customer_name,manual_risk_label,target
0,T001,Aster Bank,High,1
1,T002,Blue Harbor Retail,Low,0
2,T003,Cedar Lending,High,1
3,T004,Delta Credit,Low,0
4,T005,Echo Finance,High,1
5,T006,Fjord Payments,High,1
6,T007,Grove Markets,Low,0
7,T008,Harbor Loans,High,1


## Train the model


In [4]:
X = pd.get_dummies(df[['application_type','file_depth','account_age','identity_consistency','phone_reuse','email_reuse','address_reuse','device_reuse','ip_risk','synthetic_indicator']], drop_first=False)
y = df['target']
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X, y)
df['predicted'] = clf.predict(X)
df['predicted_label'] = df['predicted'].map({0:'Low',1:'Potentially synthetic'})
df[['application_id','customer_name','manual_risk_label','predicted_label']]


,application_id,customer_name,manual_risk_label,predicted_label
0,T001,Aster Bank,High,Potentially synthetic
1,T002,Blue Harbor Retail,Low,Low
2,T003,Cedar Lending,High,Potentially synthetic
3,T004,Delta Credit,Low,Low
4,T005,Echo Finance,High,Potentially synthetic
5,T006,Fjord Payments,High,Potentially synthetic
6,T007,Grove Markets,Low,Low
7,T008,Harbor Loans,High,Potentially synthetic


## Decision rules


In [5]:
print(export_text(clf, feature_names=list(X.columns)))


|--- ip_risk_High <= 0.50
|   |--- class: 0
|--- ip_risk_High >  0.50
|   |--- class: 1

